In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog db_novacart")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver_schema")

silver_run_id = str(uuid.uuid4())

print("Current silver_run_id : ", silver_run_id)

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS silver_schema.silver_ingestion_control(
              layer string,
              entity_name string,
              last_processed_bronze_run_id string,
              last_processed_bronze_ingested_at timestamp,
              rows_merged bigint,
              run_status string,
              silver_run_id string,
              updated_at timestamp
          )
        USING DELTA
""")

In [0]:
def upsert_to_silver(df_source, target_table, join_key):

    if spark.catalog.tableExists(target_table):
        
        dt = DeltaTable.forName(spark, target_table)
        
        dt.alias("target")\
        .merge(
                df_source.alias("source"), f"target.{join_key} = source.{join_key}"
            )\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_bronze_ingested_at(entity_name):

    ctrl = spark.table("silver_schema.silver_ingestion_control")\
        .filter(
            (col("layer") == "silver") &
            (col("entity_name") == entity_name) &
            (col("run_status") == "success")
        )\
        .orderBy(col("updated_at").desc())\
        .limit(1)

    rows = ctrl.collect()

    if not rows:
        return None
    
    else:
        return rows[0]["last_processed_bronze_ingested_at"]

In [0]:
def upsert_silver_control(entity_name, last_processed_bronze_run_id, last_processed_bronze_ingested_at, rows_merged):

    ctrl_df = spark.createDataFrame(
        [(
            "silver",
            entity_name,
            last_processed_bronze_run_id,
            last_processed_bronze_ingested_at,
            int(rows_merged),
            "success",
            silver_run_id,
            datetime.now()
        )],

        schema = """
            layer string,
            entity_name string,
            last_processed_bronze_run_id string,
            last_processed_bronze_ingested_at timestamp,
            rows_merged bigint,
            run_status string,
            silver_run_id string,
            updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, "silver_schema.silver_ingestion_control")

    dt.alias("t")\
        .merge(ctrl_df.alias("s"), "t.layer = s.layer and t.entity_name = s.entity_name")\
        .whenMatchedUpdate(
            set = {
            "last_processed_bronze_run_id": "s.last_processed_bronze_run_id",
            "last_processed_bronze_ingested_at": "s.last_processed_bronze_ingested_at",
            "rows_merged": "s.rows_merged",
            "run_status": "s.run_status",
            "silver_run_id": "s.silver_run_id",
            "updated_at": "s.updated_at"
        })\
        .whenNotMatchedInsertAll()\
        .execute()


In [0]:
def get_incrermental_bronze(bronze_table, entity_name):

    last_ingested_at = get_last_processed_bronze_ingested_at(entity_name)

    bronze_df = spark.read.table(bronze_table)

    if last_ingested_at is None:
        return bronze_df, last_ingested_at

    else:
        return bronze_df.filter(col("bronze_ingested_at") > last_ingested_at), last_ingested_at

In [0]:
df_raw = spark.sql("select distinct(order_amount) from bronze_schema.orders_raw")
display(df_raw)

In [0]:
#Read only the Bronze Order rows that silver has not processed yet.
order_inc, last_orders_ingested_at = get_incrermental_bronze("bronze_schema.orders_raw", "orders")

#Count the incremental order rows entering Silver in this run.
order_inc_count = order_inc.count()
print(f"Rows to process in silver : {order_inc_count}")

#Only run silver order cleaning and validation when there are new bronze order rows.
if order_inc_count > 0:
    #Create window that keeps the latest order record for each order_id.
    order_window = Window.partitionBy("Order_id")\
                            .orderBy(col("updated_at").desc(), col("bronze_ingested_at").desc())

    orders_cleaned = (
        order_inc
        #Standardize order_status to uppercase so values such as 'shipped' and 'SHIPPED' are consistent.
        .withColumn("order_status", upper(trim(col("order_status"))))
        .withColumn("order_status", when(col("order_status") == "", lit(None)).otherwise(col("order_status")))
        #Remove formatting characters from order_amount so it can be cast to numeric type.
        .withColumn("order_amount", regexp_replace(col("order_amount"), r"[$, ]", ""))
        .withColumn("order_amount", when(col("order_amount").isin("N/A", "NULL", "??", ""), None).otherwise(col("order_amount")))
        .withColumn("order_amount", col("order_amount").cast("double"))
        #Converting to timestamp.
        .withColumn("created_at", to_timestamp(col("created_at")))
        .withColumn("updated_at", to_timestamp(col("updated_at")))
        #Assign row number for each order_id so we can keep the latest record for each order_id.
        .withColumn("row_rank", row_number().over(order_window))
        .filter(col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", lit(silver_run_id))
    )

    #Merge the cleaned or validated silver dataset into its Delta target table.
    upsert_to_silver(orders_cleaned, "silver_schema.orders_cleaned", "order_id")

    #Apply data quality checks on silver records.
    orders_validated = (
        orders_cleaned
        .withColumn(
                    "to_be_verified_with_data_team",
                    when(col("customer_id").isNull(), "verify customer_id") 
                    .when(col("product_id").isNull(), "verify product_id")
                    .when((col("order_status").isNull()) | (trim(col("order_status")) == ""), "verify order_status")
                    .when((col("order_amount").isNull()) | (col("order_amount") <= 0), "verify order_amount")
                    .otherwise("No Issues")
                )
        .withColumn(
                    "check_order_amount",
                    when((col("order_amount").isNull()) | (col("order_amount") <= 0), lit(True)).otherwise(lit(False))
                )
        .withColumn("order_date", to_date(col("created_at")))
        .withColumn("order_year", year(col("created_at")))
        .withColumn("order_month", month(col("created_at")))
        .withColumn("order_day", dayofmonth(col("order_date")))
        .withColumn("order_dow", dayofweek(col("order_date")))
    )

    orders_good = orders_validated\
                .filter(col("to_be_verified_with_data_team") == "No Issues")

    orders_bad = orders_validated\
                .filter(col("to_be_verified_with_data_team") != "No Issues")\
                .withColumn("quarantine_ts", current_timestamp())

    #Write the good data into Delta table.
    upsert_to_silver(orders_good, "silver_schema.orders_transformed", "order_id")

    #Write the bad data into quarantine table.
    orders_bad.write.format("delta").mode("append").saveAsTable("silver_schema.orders_bad")

    #Write the last ingested timestamp into checkpoint
    max_ingested_ts = order_inc.agg(max("bronze_ingested_at").alias("max_ingested_ts")).collect()[0]["max_ingested_ts"]

    max_run_id = order_inc\
                .filter(col("bronze_ingested_at") == max_ingested_ts)\
                .agg(max("bronze_run_id").alias("max_run_id")).collect()[0]["max_run_id"]

    upsert_silver_control("orders", max_run_id, max_ingested_ts, orders_good.count())

else:
    print("No new records to process in silver.")

    upsert_silver_control("orders", None, last_orders_ingested_at, order_inc_count)

In [0]:
#Read only the Bronze Order rows that silver has not processed yet.
products_inc, last_products_ingested_at = get_incrermental_bronze("bronze_schema.products_raw", "products")

#Count the incremental order rows entering Silver in this run.
products_inc_count = products_inc.count()
print(f"Rows to process in silver : {products_inc_count}")

#Only run silver order cleaning and validation when there are new bronze order rows.
if products_inc_count > 0:
    #Create window that keeps the latest order record for each order_id.
    products_window = Window.partitionBy("product_id")\
                            .orderBy(col("updated_at").desc(), col("bronze_ingested_at").desc())

    products_cleaned = (
        products_inc
        .withColumn("product_name", upper(trim(col("product_name"))))
        .withColumn("product_name", regexp_replace(col("product_name"), r"[-_]", " "))
        .withColumn("product_name", when(col("product_name") == "", lit(None)).otherwise(col("product_name")))
        .withColumn(
            "category",
            when(upper(trim(col("category"))).contains("ELECTRNICS"), "ELECTRONICS")
            .otherwise(upper(trim(col("category"))))
        )
        .withColumn("price", trim(col("price")))
        .withColumn("price", regexp_replace(col("price"), r"\$", ""))
        .withColumn("price", regexp_replace(col("price"), r",", "."))
        .withColumn("price", regexp_replace(col("price"), r"\s+", ""))
        .withColumn("price", expr("try_cast(price as double)"))
        .withColumn("updated_at", to_timestamp(col("updated_at")))
        #Assign row number for each product_id so we can keep the latest record for each order_id.
        .withColumn("row_rank", row_number().over(products_window))
        .filter(col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", lit(silver_run_id))
    )

    #Merge the cleaned or validated silver dataset into its Delta target table.
    upsert_to_silver(products_cleaned, "silver_schema.products_cleaned", "product_id")

    #Apply data quality checks on silver records.
    products_validated = (
        products_cleaned
        .withColumn(
                    "to_be_verified_with_data_team",
                    when(col("product_name").isNull(), "verify product_name") 
                    .when(col("category").isNull(), "verify category")
                    .when((col("price").isNull()) | (col("price") <= 0), "verify price")
                    .otherwise("No Issues")
                )
        .withColumn(
                    "check_product_price",
                    when((col("price").isNull()) | (col("price") <= 0), "invalid price").otherwise("valid price")
                )
    )

    products_good = products_validated\
                .filter(
                    (col("to_be_verified_with_data_team") == "No Issues") & 
                    (col("check_product_price") == "valid price")
                )
    
    if "price_raw" in products_good.columns:
        products_good = products_good.drop("price_raw")

    products_bad = products_validated\
                .filter(
                    (col("to_be_verified_with_data_team") != "No Issues") |
                    (col("check_product_price") == "invalid price")
                    ).withColumn("quarantine_ts", current_timestamp())

    #Write the good data into Delta table.
    upsert_to_silver(products_good, "silver_schema.products_transformed", "product_id")

    #Write the bad data into quarantine table.
    products_bad.write.format("delta").mode("append").saveAsTable("silver_schema.products_bad")

    #Write the last ingested timestamp into checkpoint
    max_ingested_ts = products_inc.agg(max("bronze_ingested_at").alias("max_ingested_ts")).collect()[0]["max_ingested_ts"]

    max_run_id = products_inc\
                .filter(col("bronze_ingested_at") == max_ingested_ts)\
                .agg(max("bronze_run_id").alias("max_run_id")).collect()[0]["max_run_id"]

    upsert_silver_control("products", max_run_id, max_ingested_ts, products_good.count())

else:
    print("No new records to process in silver.")

    upsert_silver_control("products", None, last_products_ingested_at, products_inc_count)